# About

- **Author: [Dazbo](https://github.com/derailed-dash)**
- **Purpose: This is a template Python notebook for building Advent of Code solutions. Feel free to make a copy of this template for your own use. But if you use it and then share your solutions, please do keep this section to maintain the links back to my repo.**
- **Repo: [Dazbo's Advent-of-Code (All Years)](https://github.com/derailed-dash/Advent-of-Code)**

## Notes

- The source for this notebook lives <a href="https://github.com/derailed-dash/Advent-of-Code/blob/master/src/template_folder/Dazbo's_Advent_of_Code_Template.ipynb" target="_blank">here</a>.
- You can run this Notebook wherever you like. For example, you could...
  - Run it locally, in your own Jupyter environment.
  - Run it in a cloud-based Jupyter environment, with no setup required on your part!  For example, with **Google Colab**: <br><br><a href="https://colab.research.google.com/github/derailed-dash/Advent-of-Code/blob/master/src/template_folder/Dazbo's_Advent_of_Code_Template.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Google Colab"/></a><br>
  - For more ways to run Jupyter Notebooks, check out [my guide](https://medium.com/python-in-plain-english/five-ways-to-run-jupyter-labs-and-notebooks-23209f71e5c0).
- **To run the notebook, execute the cells in the [Setup](#Setup) section, as described below. Then you can run the code for any given day. But before you do, check the section below on [Getting Your AoC Data](#getting-your-aoc-data).**
- Use the navigation menu on the left to jump to any particular day.
- All of my AoC solutions are documented in my <a href="https://aoc.just2good.co.uk/" target="_blank">AoC Python Walkthrough site</a>.

## External Dependencies and Pre-Reqs

The easiest way to install the required dependencies locally is to run `make install` (or `uv sync --dev --extra jupyter`) from the repository root before launching the notebook:

```bash
make install   # runs: uv sync --dev --extra jupyter --extra legacy
```

Because [`pyproject.toml`](../../pyproject.toml) packages `src/aoc_common` editably into `.venv`, `import aoc_common.aoc_commons as ac` works automatically in local Jupyter kernels without setting `PYTHONPATH=src`. In cloud environments such as **Google Colab**, run the package install cell below to install `aoc_common` directly from GitHub.

Additionally, you may need to install required OS packages for some of the solutions, e.g. for visualisations and animations.

- [ffmpeg](https://ffmpeg.org/): in order to render video output, i.e. for visualisations.
- graphviz: for visualising graphs

In Linux, you may need to install packages manually, as the notebook will often not have necessary permissions to install dependent packages. E.g.

```bash
sudo apt install ffmpeg
sudo apt install graphviz
```

## Getting Your AoC Data

Be mindful that the first time you run this notebook, you will need to **obtain your AoC session key** and store it, if you have not done so already. This allows the notebook to automatically retrieve your input data from the AoC site, e.g. from a URL like:

`https://adventofcode.com/2026/day/1/input`

_Bear in mind that input data is unique to you._

You can find your session cookie by opening developer tools in your browser, and finding it here:

<img src="https://aoc.just2good.co.uk/assets/images/session_cookie.png" width="420" />

You can store this session key in an environment variable called `AOC_SESSION_COOKIE`. But I would recommend creating a `.env`, in the same folder you run your notebook from, which looks like this:

```text
AOC_SESSION_COOKIE=53616abcdefghi12345...
```

In Google Colab, the `.env` file will go here:

<img src="https://aoc.just2good.co.uk/assets/images/colab-dot-env.png" width="420" alt=".env file in Google Colab" />

Then the notebook will automatically pick up your session key from here.

# Setup

You need to run all cells in this section, before running any particular day solution. 

Bear in mind that if you're installing packages for the first time, you will probably need to restart the Jupyter environment/kernel before all imports will resolve. (The notebook advises when you should do this.)

## Packages and Imports

### Installing Packages in a Cloud Environment Like Google Colab

If you've setup dependencies in your virtual environment using `uv` outside the notebook, **then you don't need to run the cell below.**


In [ ]:
%pip install uv

!uv pip install \
    "git+https://github.com/derailed-dash/Advent-of-Code.git" \
    numpy \
    pandas \
    scipy \
    networkx \
    matplotlib \
    parsimonious \
    python-dotenv \
    requests \
    rich \
    tqdm \
    pillow \
    imageio \
    jupyterlab-lsp \
    ipykernel \
    ipywidgets \
    sympy \
    graphviz

### Optional Restart

If you've just installed any new packages, now would be a good time to restart the kernel!

### Core Imports

In [ ]:
# Python core imports
from __future__ import annotations  # noqa: F404

import ast
import copy
import heapq
import logging
import math
import operator
import os
import platform
import re
import subprocess
from collections import Counter, defaultdict, deque
from collections.abc import Iterable, Iterator
from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict, dataclass, field
from enum import Enum, auto
from functools import cache, lru_cache, reduce
from itertools import combinations, count, cycle, groupby, permutations, product
from pathlib import Path

# Third-party imports
import graphviz
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import sympy
from dotenv import load_dotenv
from IPython.display import Image, Markdown, Video, display
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon, Rectangle
from tqdm.auto import tqdm

# Shared AoC helpers
import aoc_common.aoc_commons as ac

## Logging and Output

Configure logging using `ac.setup_logging()` (`RichHandler`), which formats coloured output cleanly in both terminal scripts and Jupyter notebooks.

In [ ]:
# Setup logger
YEAR = 2026
APP_NAME = f"aoc{YEAR}"
ac.setup_logging(level=logging.DEBUG)
logger = logging.getLogger(APP_NAME)
logger.setLevel(logging.DEBUG)
logger.info("Logger initialised.")
logger.debug("Debugging enabled")

## Shared AoC Helpers (`aoc_common`)

All shared AoC helpers are imported from [`src/aoc_common/aoc_commons.py`](../aoc_common/aoc_commons.py) via `import aoc_common.aoc_commons as ac` so standalone scripts and notebooks share a single source of truth:

- **Locations & input retrieval**: `Locations`, `get_locations`, `write_puzzle_input_file`, `top_and_tail`
- **Validation & timing**: `validate`, `timer`
- **2D grids & geometry**: `Point`, `Vectors`, `VectorDicts`, `Grid`
- **Algorithms**: `binary_search`, `merge_intervals`, `get_factors`, `to_base_n`

Below we define only the notebook-specific `show_day_link` helper for rendering clickable day headings.

In [ ]:
def show_day_link(day: int | str) -> None:
    """Render a Markdown link to the Advent of Code puzzle page for ``day``."""
    day_link = f"#### See [Day {day}](https://adventofcode.com/{YEAR}/day/{day})."
    display(Markdown(day_link))

# Env Clear

Only run the next cell if you want to manually clear your session key.

In [ ]:
del os.environ['AOC_SESSION_COOKIE']

# Days

Here you'll find a template to build a solution for a given day, and then the solutions for all days in this event.

To copy the template day, select all the cells in the `Day n` template, add a new cell at the end, and then paste the cells there.

---
## Day n: title

In [ ]:
DAY = "1" # replace with actual number (without leading digit)
show_day_link(DAY)

In [ ]:
d_name = "d" + str(DAY).zfill(2) # e.g. d01
script_name = "aoc" + str(YEAR) + d_name # e.g. aoc2026d01
locations = ac.get_locations(script_name, d_name)
logger.setLevel(logging.DEBUG)

# Retrieve input and store in local file
try:
    ac.write_puzzle_input_file(YEAR, int(DAY), locations)
    with open(locations.input_file, encoding="utf-8") as f:
        input_data = f.read().splitlines()

    logger.info("Input data:\n%s", ac.top_and_tail(input_data))
except ValueError as e:
    logger.error(e)

### Day n Part 1

Overview...

In [ ]:
def solve_part1(data):
    pass

In [ ]:
%%time
sample_inputs = []
sample_inputs.append("""abcdef""")
sample_answers = ["uvwxyz"]

logger.setLevel(logging.DEBUG)
for curr_input, curr_ans in zip(sample_inputs, sample_answers):
    ac.validate(solve_part1(curr_input), curr_ans) # test with sample data
    logger.info("Test passed")

logger.info("All tests passed!")

logger.setLevel(logging.INFO)
soln = solve_part1(input_data)
logger.info(f"Part 1 soln={soln}")

### Day n Part 2

Overview...

In [ ]:
def solve_part2(data):
    pass

In [ ]:
%%time
sample_inputs = []
sample_inputs.append("""abcdef""")
sample_answers = ["uvwxyz"]

logger.setLevel(logging.DEBUG)
for curr_input, curr_ans in zip(sample_inputs, sample_answers):
    ac.validate(solve_part2(curr_input), curr_ans) # test with sample data
    logger.info("Test passed")

logger.info("Tests passed!")

logger.setLevel(logging.INFO)
soln = solve_part2(input_data)
logger.info(f"Part 2 soln={soln}")